# 03 — Feature Engineering
Build the full feature set used for model training: occupancy metrics, weather, events, time encoding.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA = Path('../data')
VIZ  = Path('../visualizations')

In [ ]:
features = pd.read_csv(DATA / 'features.csv', parse_dates=['hour'])
print(f'Feature set: {features.shape[0]:,} rows × {features.shape[1]} columns')
features.head()

## Feature Groups

In [ ]:
groups = {
    'Target':        ['avg_occupancy_rate'],
    'Parking':       ['peak_occupancy_rate', 'total_spaces', 'total_occupied', 'num_blockfaces', 'turnover_proxy'],
    'Weather':       ['temperature', 'precipitation', 'wind_speed', 'elevation'],
    'Events':        ['is_event_day', 'event_count', 'has_city_event', 'max_attendance', 'has_road_closure'],
    'Holiday':       ['is_holiday', 'holiday_name'],
    'Time (raw)':    ['hour_of_day', 'day_of_week', 'month', 'year', 'is_weekend', 'is_peak_am', 'is_peak_pm'],
    'Time (cyclic)': ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos'],
}

for group, cols in groups.items():
    available = [c for c in cols if c in features.columns]
    print(f'{group:<16} {len(available)}/{len(cols)} features: {available}')

## Correlation with Target (Occupancy Rate)

In [ ]:
numeric_features = features.select_dtypes(include='number')
corr_with_target = numeric_features.corr()['avg_occupancy_rate'].drop('avg_occupancy_rate').sort_values()

plt.figure(figsize=(10, 8))
colors = ['#EF5350' if v < 0 else '#42A5F5' for v in corr_with_target.values]
corr_with_target.plot(kind='barh', color=colors)
plt.title('Feature Correlation with Occupancy Rate')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(VIZ / 'model/feature_correlation_with_target.png', dpi=150)
plt.show()

## Cyclical Time Encoding Explained

In [ ]:
hours = np.arange(0, 24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hours, hour_sin, 'b-o', markersize=4, label='sin')
axes[0].plot(hours, hour_cos, 'r-o', markersize=4, label='cos')
axes[0].set_title('Cyclical Hour Encoding (sin/cos)')
axes[0].set_xlabel('Hour of Day')
axes[0].legend()

axes[1].scatter(hour_sin, hour_cos, c=hours, cmap='hsv', s=60)
for h in [0, 6, 12, 18]:
    axes[1].annotate(f'{h}:00', (hour_sin[h], hour_cos[h]), textcoords='offset points', xytext=(5,5))
axes[1].set_title('Hour Encoded as Circle (no discontinuity at midnight)')
axes[1].set_xlabel('sin(hour)')
axes[1].set_ylabel('cos(hour)')
plt.tight_layout()
plt.savefig(VIZ / 'model/cyclical_encoding.png', dpi=150)
plt.show()

## Feature Set Summary

In [ ]:
print(f'Total features available: {features.shape[1]}')
print(f'Rows (hourly region snapshots): {features.shape[0]:,}')
print()
print('Missing values:')
print(features.isnull().sum()[features.isnull().sum() > 0])
print()
print('Feature dtypes:')
print(features.dtypes.value_counts())